**What this notebook does:**
1. Cleans up the datasets for data visualization.


2. Creates a z-score column relative to the company's specific category.
3. Creates a 95% confidence interval for the probable average ratings a company could have.
4. Creates 2 datasets that can be used with these new columns of different sizes.

**Step 1:** clean datasets before merging and feature engineering, so Tableau is displayed without outliers and NULL data.

In [124]:
import pandas as pd
import numpy as np
import gzip
import json

pd.set_option('display.max_columns', None)

def parse(path):
  g = open('Data/' + path, 'r')
  for l in g:
    yield json.loads(l)

def parse_first_n(path, n=10000):
    g = open('Data/' + path, 'r')
    for i, l in enumerate(g):
        if i >= n:
            break
        yield json.loads(l)

In [125]:
texas_metadata = pd.DataFrame(parse("meta-Texas.json"))
texas_reviews = pd.DataFrame(parse_first_n("review-Texas.json", n=1000000)) # is not cleaned yet
#add more or less reviews depending on how well tableu responds to the data.


KeyboardInterrupt: 

Exception ignored in: 'zmq.backend.cython._zmq.Frame.__dealloc__'
Traceback (most recent call last):
  File "zmq/backend/cython/_zmq.py", line 179, in zmq.backend.cython._zmq._check_rc
KeyboardInterrupt: 


In [126]:
texas_reviews = texas_reviews.dropna(subset=['rating']) # Drop rows where 'rating' is NaN

In [127]:
texas_metadata = texas_metadata.dropna(subset=['gmap_id', 'name']).drop_duplicates(subset=['gmap_id'], keep='first')



In [128]:
texas_metadata = texas_metadata[texas_metadata['state'] != 'Permanently closed']


In [129]:
texas_metadata = texas_metadata[(texas_metadata['latitude'] >= 25.5) & (texas_metadata['latitude'] <= 36.40)
                            & (texas_metadata['longitude'] >= -106.50) & (texas_metadata['longitude'] <= -93.31)]


In [130]:
texas_metadata.shape[0]

416224

In [131]:
path = 'Data/simplemaps_uscities_basicv1.93/uscities.csv'

us_cities = pd.read_csv(path)

import numpy as np
from sklearn.neighbors import BallTree

# Convert to radians


category_df = texas_metadata.copy()
#THIS WILL FIRST ADD A CITY COLUMN TO OUR METADATA SO WE CAN GROUP
cities_rad = np.radians(us_cities[['lat', 'lng']])
companies_rad = np.radians(category_df[['latitude', 'longitude']])

# Build BallTree
tree = BallTree(cities_rad, metric='haversine')

# Find nearest city for each company
distances, indices = tree.query(companies_rad, k=1)

# Assign city name
nearest_city_names = us_cities.iloc[indices.flatten()]['city'].reset_index(drop=True)

category_df = category_df.reset_index(drop=True)
category_df['city'] = nearest_city_names
#Extract all unique category labels from california_metadata
category_df = category_df.explode('category')
#Convert everything to lower case and remove blank space 
category_df['category'] = category_df['category'].str.lower().str.strip()

# # # #Remove tiny categories 
category_counts= category_df.get('category').value_counts()


# Setting threshold for the category 
category_counts.describe()
category_counts_threshold = 35 #median, we could alter this if there are too many categories for our deomo


# Map category_count to each category in the df
category_df['category_count'] = category_df['category'].map(category_counts)
category_df = category_df[category_df.get('category_count') > category_counts_threshold]
category_df = category_df.sort_values(by='category_count', ascending=False)




In [132]:

# Calculate mean, std, and count for each category within each city
category_stats = category_df.groupby(["city", "category"])['avg_rating'].agg(['mean', 'std', 'count']).reset_index()
category_stats.columns = ['city', 'category', 'cc_mean', 'cc_std', 'cc_count']  # Flatten the MultiIndex columns

In [133]:

# Merge back into category data set
mcategory_df = category_df.merge(category_stats, on=['city', 'category'], how='inner') #merge on both city and category
mcategory_df = mcategory_df.rename(columns={
    'cc_mean': 'category_mean',
    'cc_std': 'category_std'
})
mcategory_df.shape[0]

952062

In [134]:
mcategory_df.columns

Index(['name', 'address', 'gmap_id', 'description', 'latitude', 'longitude',
       'category', 'avg_rating', 'num_of_reviews', 'price', 'hours', 'MISC',
       'state', 'relative_results', 'url', 'city', 'category_count',
       'category_mean', 'category_std', 'cc_count'],
      dtype='object')

In [135]:

# competition score = normalized rating in local area relative to competition
# If less than 2 companies in the city-category combo, mark as insufficient data
mcategory_df['competition_score'] = mcategory_df.apply(
    lambda row: 
        (row['avg_rating'] - row['category_mean']) / row['category_std']
        if (row['cc_count'] >= 2 and row['category_std'] != 0)
        else np.nan,
    axis=1
)
mcategory_df #now this is going to include repeats of companies when it is done

mcategory_df = mcategory_df.groupby('gmap_id').agg({
    'competition_score': 'first'  # Changed from 'mean' to 'first' since now we have mixed types (float and string)
}).reset_index()

mcategory_df #The reason that the number dropped is becuase explode will create multiple rows and grouping by gmap_id will reduce that back

,gmap_id,competition_score
0,0x0:0x3de3e25d19fc7da6,-0.105706
1,0x0:0x48c466d0f3de5f5a,0.408221
2,0x0:0x554de8957fed30a1,0.038640
3,0x0:0x5702eb6d2d1301db,1.333021
4,0x1000000000000001:0xe85d5b0e25befcb7,1.292591
...,...,...
409628,0xaceec0ea6c38bcfd:0xf291bc79aa31fa60,0.707107
409629,0xad33c203e03d89f9:0xb8ee8c778a581ca4,0.519618
409630,0xd0c7e1a240869ad:0xafa498adc8911c71,0.567712
409631,0xdce6e3016b65401:0xf220f369359026f2,0.676668


In [136]:
#We will merge all of this back into texas metadata with an inner merge to add the new z-score and CI columns for the Tableu dataset.

company_stats = (
    texas_reviews #first time using texas reviews, we should definetly do a left merge for CI, since limmited data n=1000000
        .groupby('gmap_id')['rating']
        .agg(['mean', 'std', 'count'])
        .reset_index()
)

def compute_ci(row):
    mean = row['mean']
    std = row['std']
    n = row['count']
    
    if n == 0:
        return None
    
    margin = 1.96 * (std / np.sqrt(n))
    return (mean - margin, mean + margin)

company_stats['CI'] = company_stats.apply(compute_ci, axis=1)

mtexas_metadata = texas_metadata.merge(
    company_stats[['gmap_id', 'CI']],
    on='gmap_id',
    how='inner'
)

# mcategory_df['CI']= mcategory_df['gmap_id'].apply(individual_company_rating_confidence_interval)
# mcategory_df #THIS TAKES TO LONG SPEED IT UP AND THEN MERGE BACK TO TEXAS METADATA



In [137]:


final_texas_metadata = texas_metadata.merge(mtexas_metadata[['gmap_id', 'CI']], on='gmap_id', how='left').merge(mcategory_df[['gmap_id', 'competition_score']], on='gmap_id', how='left')
final_texas_metadata['competition_score'] = final_texas_metadata['competition_score'].fillna("not enough data")
#shuold be > 409k

****BELOW ARE 2 FINAL DATASETS. ONE WITH ONLY THE COMPANIES IN TEXAS REVIEWS WITH NO NAN VALUES this is small_... AND THE OTHER INCLUDES ALL THE DATA IN META BUT NAN VALUES. this is big_.... Choose which one to use BASED ON HOW WELL TABLEAU HANDLES A LOT OF DATA.****

In [138]:
small_test_final_texas_metadata = final_texas_metadata.dropna(subset=['CI'])
small_test_final_texas_metadata = small_test_final_texas_metadata[~small_test_final_texas_metadata['CI'].apply(lambda x: any(pd.isna(i) for i in x))]
small_test_final_texas_metadata['CI'] = small_test_final_texas_metadata['CI'].apply(lambda x: (x[0],5.0) if x[1] > 5.0 else (1.0, x[1]) if x[0] < 1.0 else x)
small_test_final_texas_metadata = small_test_final_texas_metadata.rename(columns={
    'CI': 'CI_for_avg_rating',
    'rating_z-score': 'category_relative_score'
})
small_test_final_texas_metadata


,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,CI_for_avg_rating,competition_score
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,None,30.713368,-94.954344,[Convenience store],4.8,4,None,"[[Thursday, Open 24 hours], [Friday, Open 24 h...","{'Service options': ['In-store shopping', 'Del...",Open 24 hours,"[0x863886bab3f9bb05:0x28a8062d0597dd34, 0x8638...",https://www.google.com/maps/place//data=!4m2!3...,"(4.429219701353091, 5.0)",1.696803
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,None,33.159363,-96.975571,[Transportation service],4.5,4,None,"[[Thursday, 6AM–6PM], [Friday, 6AM–6PM], [Satu...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 6PM,"[0x864c374855555555:0x3abb669a098bb235, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(3.858439402706183, 5.0)",not enough data
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,"[Pharmacy, Drug store, Medical supply store, V...",3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(2.773104882616715, 3.810228450716618)",1.1094
3,Luminous Logistics,"Luminous Logistics, 3838 W Miller Rd, Garland,...",0x864ea0993bffffff:0xb50b5bb2fccf9d9b,None,32.893678,-96.688611,[Delivery service],2.3,8,None,None,None,None,"[0x864ea09938bb619f:0x1b6902de2a2f3f96, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.382640020906352, 3.117359979093648)",not enough data
4,Pacesetter Personnel Services,"Pacesetter Personnel Services, 2300 Valley Vie...",0x864e819d99a1ff99:0xeee31cc82854286c,None,32.839795,-97.020987,[Employment agency],2.1,7,None,None,None,None,"[0x864e9d6ea0c9089f:0x6f90f8b0b092af49, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.16055216123327, 3.1251621244810153)",-1.696591
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36880,Larry Pepper A/C & Heating Inc,"Larry Pepper A/C & Heating Inc, 1810 Andrews H...",0x86fbc96cddee47e5:0xe5edb0ba703c33f9,None,31.860806,-102.377652,"[HVAC contractor, Air conditioning repair serv...",3.8,28,None,"[[Friday, 8AM–5PM], [Saturday, Closed], [Sunda...",{'Offerings': ['Repair services']},Open ⋅ Closes 5PM,"[0x86fbc9af1d98a9f5:0xb5e117d567a94ef3, 0x86fb...",https://www.google.com/maps/place//data=!4m2!3...,"(3.1834137103454125, 4.459443432511731)",-0.837699
36881,La Casita Antiques,"La Casita Antiques, 3807 W County Rd 118, Midl...",0x86fbd786f3f4687d:0xf47d2c838c940850,None,31.951504,-102.109351,"[Antique furniture store, Antique store]",3.7,3,None,"[[Friday, Closed], [Saturday, 10:30AM–5:30PM],...","{'Service options': ['In-store shopping'], 'Pl...",Closed ⋅ Opens 10:30AM Sat,"[0x86fbd87949c1eaa5:0xc75b5798311d6628, 0x86fb...",https://www.google.com/maps/place//data=!4m2!3...,"(3.013333333333333, 4.32)",-1.591645
36883,The Conner Team with City First Mortgage Services,The Conner Team with City First Mortgage Servi...,0x86fe712c76442f31:0x83b8a33b9af45707,None,33.490310,-101.936153,[Mortgage lender],5.0,18,None,"[[Friday, 9AM–5:30PM], [Saturday, Closed], [Su...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 5:30PM,"[0x86fe71859c30e2f7:0x98556f8d47fe6027, 0x86fe...",https://www.google.com/maps/place//data=!4m2!3...,"(5.0, 5.0)",1.14348
36884,J & B Plumbing,"J & B Plumbing, 7502 N County Rd 1297, Midland...",0x86fbc36f09935b9b:0x9a2b804b6896fd9e,None,32.040941,-102.286099,[Plumber],5.0,4,None,None,None,Open now,"[0x86fbdb37f6179ec1:0x503944e7edaf5433, 0x86fb...",https://www.google.com/maps/place//data=!4m2!3...,"(5.0, 5.0)",not enough data


In [139]:
big_test_final_texas_metadata = final_texas_metadata.copy()

def clamp_ci(x): #this fucniton is only needed for big due to NaN handling
    # Skip NaN or non-tuples
    if not isinstance(x, (tuple, list)):
        return x
    
    lower, upper = x
    
    # Clamp values
    lower = max(1.0, lower)
    upper = min(5.0, upper)
    
    return (lower, upper)

big_test_final_texas_metadata['CI'] = (
    big_test_final_texas_metadata['CI'].apply(clamp_ci)
)

# Rename columns
big_test_final_texas_metadata = big_test_final_texas_metadata.rename(columns={
    'CI': 'CI_for_avg_rating',
    'rating_z-score': 'category_relative_score'
})


In [140]:
print(big_test_final_texas_metadata.shape[0], small_test_final_texas_metadata.shape[0]) #big vs small difference

416224 34376


In [141]:
big_test_final_texas_metadata

,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,CI_for_avg_rating,competition_score
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,None,30.713368,-94.954344,[Convenience store],4.8,4,None,"[[Thursday, Open 24 hours], [Friday, Open 24 h...","{'Service options': ['In-store shopping', 'Del...",Open 24 hours,"[0x863886bab3f9bb05:0x28a8062d0597dd34, 0x8638...",https://www.google.com/maps/place//data=!4m2!3...,"(4.429219701353091, 5.0)",1.696803
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,None,33.159363,-96.975571,[Transportation service],4.5,4,None,"[[Thursday, 6AM–6PM], [Friday, 6AM–6PM], [Satu...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 6PM,"[0x864c374855555555:0x3abb669a098bb235, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(3.858439402706183, 5.0)",not enough data
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,"[Pharmacy, Drug store, Medical supply store, V...",3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(2.773104882616715, 3.810228450716618)",1.1094
3,Luminous Logistics,"Luminous Logistics, 3838 W Miller Rd, Garland,...",0x864ea0993bffffff:0xb50b5bb2fccf9d9b,None,32.893678,-96.688611,[Delivery service],2.3,8,None,None,None,None,"[0x864ea09938bb619f:0x1b6902de2a2f3f96, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.382640020906352, 3.117359979093648)",not enough data
4,Pacesetter Personnel Services,"Pacesetter Personnel Services, 2300 Valley Vie...",0x864e819d99a1ff99:0xeee31cc82854286c,None,32.839795,-97.020987,[Employment agency],2.1,7,None,None,None,None,"[0x864e9d6ea0c9089f:0x6f90f8b0b092af49, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.16055216123327, 3.1251621244810153)",-1.696591
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
416219,Texture,"Texture, 2505 W SW Loop 323, Tyler, TX 75701",0x8649ceaecd6cd113:0xcb1ded587cb53b35,None,32.305068,-95.332749,"[Interior designer, Fabric store]",4.9,8,None,"[[Wednesday, 9:30AM–5PM], [Thursday, 9:30AM–5P...",None,NaN,"[0x8649ccfe591fd837:0x650fe45e1c1b0dd7, 0x8649...",https://www.google.com/maps/place//data=!4m2!3...,NaN,0.19868
416220,Sheraton Austin Georgetown Hotel & Conference ...,Sheraton Austin Georgetown Hotel & Conference ...,0x8644d53fe5245059:0x793047042b121b1e,"Refined rooms in an upscale hotel, offering a ...",30.649195,-97.686135,"[Hotel, Indoor lodging, Meeting planning servi...",4.6,948,None,None,None,NaN,None,https://www.google.com/maps/place//data=!4m2!3...,NaN,1.190887
416221,Hana Travel Plaza,"Hana Travel Plaza, 11301 I-40, Amarillo, TX 79118",0x870137dffb31b41b:0xea3a20936f410b50,None,35.193626,-101.706485,"[Truck stop, Truck accessories store]",4.1,808,None,"[[Wednesday, Open 24 hours], [Thursday, Open 2...","{'Service options': ['In-store shopping', 'Del...",NaN,"[0x870148f39aeb5bef:0x669fd06662a0d050, 0x8701...",https://www.google.com/maps/place//data=!4m2!3...,NaN,-1.168699
416222,Pilot Travel Center,"Pilot Travel Center, 100 S Poplar St, Stratfor...",0x8705dff80d5a8531:0xcfd4e9fa1f92f376,None,36.331260,-102.073430,"[Truck stop, Convenience store, Gas station]",3.9,1593,None,"[[Wednesday, Open 24 hours], [Thursday, Open 2...","{'Service options': ['In-store shopping', 'Del...",NaN,"[0x8705dff878be024f:0xb42ee8fe9f7898be, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,NaN,-0.707107


In [142]:
path = 'Data/simplemaps_uscities_basicv1.93/uscities.csv'

us_cities = pd.read_csv(path)

import numpy as np
from sklearn.neighbors import BallTree

# Convert to radians
cities_rad = np.radians(us_cities[['lat', 'lng']])
companies_rad = np.radians(big_test_final_texas_metadata[['latitude', 'longitude']])

# Build BallTree
tree = BallTree(cities_rad, metric='haversine')

# Find nearest city for each company
distances, indices = tree.query(companies_rad, k=1)

# Assign city name
nearest_city_names = us_cities.iloc[indices.flatten()]['city'].reset_index(drop=True)

big_test_final_texas_metadata = big_test_final_texas_metadata.reset_index(drop=True)
big_test_final_texas_metadata['city'] = nearest_city_names # uses city df for indicies and company df for assign
#.values is used so it assigns the city in the correct order to the company.
#this is only going to work after we asign city to each company
def standardize_rating_by_city(df):
    """
    Standardizes average ratings relative to each city's mean and standard deviation.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame with 'avg_rating' and 'city' columns
        
    Returns:
    --------
    pandas.Series
        Z-scores (standardized ratings) for each row
    """
    # Calculate city-level statistics
    city_stats = df.groupby('city')['avg_rating'].agg(['mean', 'std'])
    
    # Map city statistics to each row
    df_with_stats = df.copy()
    df_with_stats['city_mean'] = df['city'].map(city_stats['mean'])
    df_with_stats['city_std'] = df['city'].map(city_stats['std']) #map is like a mini merge
    
    # Calculate z-score: (value - mean) / std
    # Handle cases where std is 0 (all ratings in city are the same)
    standardized = (df_with_stats['avg_rating'] - df_with_stats['city_mean']) / df_with_stats['city_std'].replace(0, 1)
    
    return standardized

# Apply the function to add standardized ratings
big_test_final_texas_metadata['standardized_city_rating'] = standardize_rating_by_city(big_test_final_texas_metadata)

In [143]:
big_test_final_texas_metadata #this is a good sign that the city merge worked, we have a lot of companies in houston which is what we would expect.

,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,CI_for_avg_rating,competition_score,city,standardized_city_rating
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,None,30.713368,-94.954344,[Convenience store],4.8,4,None,"[[Thursday, Open 24 hours], [Friday, Open 24 h...","{'Service options': ['In-store shopping', 'Del...",Open 24 hours,"[0x863886bab3f9bb05:0x28a8062d0597dd34, 0x8638...",https://www.google.com/maps/place//data=!4m2!3...,"(4.429219701353091, 5.0)",1.696803,Livingston,0.972932
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,None,33.159363,-96.975571,[Transportation service],4.5,4,None,"[[Thursday, 6AM–6PM], [Friday, 6AM–6PM], [Satu...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 6PM,"[0x864c374855555555:0x3abb669a098bb235, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(3.858439402706183, 5.0)",not enough data,Lakewood Village,0.243142
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,"[Pharmacy, Drug store, Medical supply store, V...",3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(2.773104882616715, 3.810228450716618)",1.1094,Little Elm,-1.271692
3,Luminous Logistics,"Luminous Logistics, 3838 W Miller Rd, Garland,...",0x864ea0993bffffff:0xb50b5bb2fccf9d9b,None,32.893678,-96.688611,[Delivery service],2.3,8,None,None,None,None,"[0x864ea09938bb619f:0x1b6902de2a2f3f96, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.382640020906352, 3.117359979093648)",not enough data,Garland,-2.768392
4,Pacesetter Personnel Services,"Pacesetter Personnel Services, 2300 Valley Vie...",0x864e819d99a1ff99:0xeee31cc82854286c,None,32.839795,-97.020987,[Employment agency],2.1,7,None,None,None,None,"[0x864e9d6ea0c9089f:0x6f90f8b0b092af49, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.16055216123327, 3.1251621244810153)",-1.696591,Irving,-2.680228
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
416219,Texture,"Texture, 2505 W SW Loop 323, Tyler, TX 75701",0x8649ceaecd6cd113:0xcb1ded587cb53b35,None,32.305068,-95.332749,"[Interior designer, Fabric store]",4.9,8,None,"[[Wednesday, 9:30AM–5PM], [Thursday, 9:30AM–5P...",None,NaN,"[0x8649ccfe591fd837:0x650fe45e1c1b0dd7, 0x8649...",https://www.google.com/maps/place//data=!4m2!3...,NaN,0.19868,Tyler,0.989413
416220,Sheraton Austin Georgetown Hotel & Conference ...,Sheraton Austin Georgetown Hotel & Conference ...,0x8644d53fe5245059:0x793047042b121b1e,"Refined rooms in an upscale hotel, offering a ...",30.649195,-97.686135,"[Hotel, Indoor lodging, Meeting planning servi...",4.6,948,None,None,None,NaN,None,https://www.google.com/maps/place//data=!4m2!3...,NaN,1.190887,Georgetown,0.389662
416221,Hana Travel Plaza,"Hana Travel Plaza, 11301 I-40, Amarillo, TX 79118",0x870137dffb31b41b:0xea3a20936f410b50,None,35.193626,-101.706485,"[Truck stop, Truck accessories store]",4.1,808,None,"[[Wednesday, Open 24 hours], [Thursday, Open 2...","{'Service options': ['In-store shopping', 'Del...",NaN,"[0x870148f39aeb5bef:0x669fd06662a0d050, 0x8701...",https://www.google.com/maps/place//data=!4m2!3...,NaN,-1.168699,Amarillo,-0.275990
416222,Pilot Travel Center,"Pilot Travel Center, 100 S Poplar St, Stratfor...",0x8705dff80d5a8531:0xcfd4e9fa1f92f376,None,36.331260,-102.073430,"[Truck stop, Convenience store, Gas station]",3.9,1593,None,"[[Wednesday, Open 24 hours], [Thursday, Open 2...","{'Service options': ['In-store shopping', 'Del...",NaN,"[0x8705dff878be024f:0xb42ee8fe9f7898be, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,NaN

In [144]:
big_test_final_texas_metadata.isna().sum()
#city is not null at all, so we are good to group by it

name                             0
address                       8551
gmap_id                          0
description                 340106
latitude                         0
longitude                        0
category                      1574
avg_rating                       0
num_of_reviews                   0
price                       340669
hours                        85151
MISC                         67655
state                       117313
relative_results             37575
url                              0
CI_for_avg_rating           379338
competition_score                0
city                             0
standardized_city_rating        67
dtype: int64

In [145]:
big_test_final_texas_metadata

,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,CI_for_avg_rating,competition_score,city,standardized_city_rating
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,None,30.713368,-94.954344,[Convenience store],4.8,4,None,"[[Thursday, Open 24 hours], [Friday, Open 24 h...","{'Service options': ['In-store shopping', 'Del...",Open 24 hours,"[0x863886bab3f9bb05:0x28a8062d0597dd34, 0x8638...",https://www.google.com/maps/place//data=!4m2!3...,"(4.429219701353091, 5.0)",1.696803,Livingston,0.972932
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,None,33.159363,-96.975571,[Transportation service],4.5,4,None,"[[Thursday, 6AM–6PM], [Friday, 6AM–6PM], [Satu...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 6PM,"[0x864c374855555555:0x3abb669a098bb235, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(3.858439402706183, 5.0)",not enough data,Lakewood Village,0.243142
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,"[Pharmacy, Drug store, Medical supply store, V...",3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,"(2.773104882616715, 3.810228450716618)",1.1094,Little Elm,-1.271692
3,Luminous Logistics,"Luminous Logistics, 3838 W Miller Rd, Garland,...",0x864ea0993bffffff:0xb50b5bb2fccf9d9b,None,32.893678,-96.688611,[Delivery service],2.3,8,None,None,None,None,"[0x864ea09938bb619f:0x1b6902de2a2f3f96, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.382640020906352, 3.117359979093648)",not enough data,Garland,-2.768392
4,Pacesetter Personnel Services,"Pacesetter Personnel Services, 2300 Valley Vie...",0x864e819d99a1ff99:0xeee31cc82854286c,None,32.839795,-97.020987,[Employment agency],2.1,7,None,None,None,None,"[0x864e9d6ea0c9089f:0x6f90f8b0b092af49, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...,"(1.16055216123327, 3.1251621244810153)",-1.696591,Irving,-2.680228
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
416219,Texture,"Texture, 2505 W SW Loop 323, Tyler, TX 75701",0x8649ceaecd6cd113:0xcb1ded587cb53b35,None,32.305068,-95.332749,"[Interior designer, Fabric store]",4.9,8,None,"[[Wednesday, 9:30AM–5PM], [Thursday, 9:30AM–5P...",None,NaN,"[0x8649ccfe591fd837:0x650fe45e1c1b0dd7, 0x8649...",https://www.google.com/maps/place//data=!4m2!3...,NaN,0.19868,Tyler,0.989413
416220,Sheraton Austin Georgetown Hotel & Conference ...,Sheraton Austin Georgetown Hotel & Conference ...,0x8644d53fe5245059:0x793047042b121b1e,"Refined rooms in an upscale hotel, offering a ...",30.649195,-97.686135,"[Hotel, Indoor lodging, Meeting planning servi...",4.6,948,None,None,None,NaN,None,https://www.google.com/maps/place//data=!4m2!3...,NaN,1.190887,Georgetown,0.389662
416221,Hana Travel Plaza,"Hana Travel Plaza, 11301 I-40, Amarillo, TX 79118",0x870137dffb31b41b:0xea3a20936f410b50,None,35.193626,-101.706485,"[Truck stop, Truck accessories store]",4.1,808,None,"[[Wednesday, Open 24 hours], [Thursday, Open 2...","{'Service options': ['In-store shopping', 'Del...",NaN,"[0x870148f39aeb5bef:0x669fd06662a0d050, 0x8701...",https://www.google.com/maps/place//data=!4m2!3...,NaN,-1.168699,Amarillo,-0.275990
416222,Pilot Travel Center,"Pilot Travel Center, 100 S Poplar St, Stratfor...",0x8705dff80d5a8531:0xcfd4e9fa1f92f376,None,36.331260,-102.073430,"[Truck stop, Convenience store, Gas station]",3.9,1593,None,"[[Wednesday, Open 24 hours], [Thursday, Open 2...","{'Service options': ['In-store shopping', 'Del...",NaN,"[0x8705dff878be024f:0xb42ee8fe9f7898be, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,NaN

With the code below we could do number of reviews in a geo cell relative to the idetified population, that would showcase how active the people are in reviewing. **MAKE THE PROPORTION** of population density to number_of_reviews in the geocell.

In [ ]:
#TODO make a relative surounding area density feature to define how dense the city is relative to its suroudnings
